# Multi-Model Threshold Evaluation (FPR=20%)

This notebook recomputes thresholded metrics for six trained autoencoders using **mean** reconstruction error with an **FPR-controlled threshold** of **0.2**.

## Scope

This notebook is designed for your current setup:
- data available as ZIP files in Drive
- trained model checkpoints (`.pth`) available in Drive

Models included:
- ECNN-AE (optimized)
- CNN-AE Large
- CNN-AE Augmented
- CNN-AE
- ResNet-AE
- ResNet Finetuned (partial)

Outputs generated:
- Confusion matrix (counts) per model
- Normalized confusion matrix per model
- ROC curve per model
- Combined ROC comparison plot
- Consolidated metrics table (CSV + Markdown)

Threshold policy (fixed):
- Score method: `mean`
- Threshold method: `fpr`
- Parameter: `0.2`

In [1]:
# Colab detection and Drive mount
import os

try:
    from google.colab import drive  # type: ignore
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
        print('Mounted Google Drive at /content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print('Running outside Colab; ensure Drive paths are accessible.')

Mounted at /content/drive
Mounted Google Drive at /content/drive


In [3]:
# Resolve repository and module paths
import sys
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/RifaDeen/symAD-ECNN.git'
CLONE_PATH = Path('/content/symAD-ECNN')

if IN_COLAB:
    if not CLONE_PATH.exists():
        print('Cloning repository...')
        subprocess.check_call(['git', 'clone', REPO_URL, str(CLONE_PATH)])
    REPO_ROOT = CLONE_PATH
else:
    REPO_ROOT = Path.cwd()
    while REPO_ROOT != REPO_ROOT.parent:
        if (REPO_ROOT / 'README.md').exists() and (REPO_ROOT / 'notebooks').exists():
            break
        REPO_ROOT = REPO_ROOT.parent
    else:
        raise RuntimeError('Unable to locate project root. Open this notebook inside the project.')

EVALS_DIR = REPO_ROOT / 'notebooks' / 'evals'
MODULE_DIRS = [REPO_ROOT, EVALS_DIR, EVALS_DIR / 'ecnn_thresholding']
for p in MODULE_DIRS:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

print(f'Repository root: {REPO_ROOT}')
print(f'Evals path: {EVALS_DIR}')

Cloning repository...
Repository root: /content/symAD-ECNN
Evals path: /content/symAD-ECNN/notebooks/evals


In [20]:
# Imports
import os
import zipfile
import shutil
from glob import glob
from pathlib import Path
from typing import Dict, Tuple, List, Optional
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from config import DRIVE_PROJECT_ROOT, EVALUATIONS_ROOT, ensure_directories_exist
from metrics_utils import threshold_from_normal_scores, compute_full_metrics
from plotting_utils import (
    plot_confusion_matrix,
    plot_roc_curve,
    plot_multiple_roc_curves,
    save_figure,
 )
from io_utils import save_csv, save_markdown_table
import ecnn_model_loader # Explicitly import the module
from ecnn_model_loader import get_model_for_inference

import importlib
importlib.reload(ecnn_model_loader) # Reload to pick up e2cnn after installation

ensure_directories_exist()
plt.style.use('seaborn-v0_8')
print('Imports and directories ready.')

Imports and directories ready.


In [22]:
!pip install e2cnn

In [23]:
PROJECT_ROOT = DRIVE_PROJECT_ROOT if DRIVE_PROJECT_ROOT.exists() else REPO_ROOT
DATA_ROOT = PROJECT_ROOT / 'data'
MODELS_ROOT = PROJECT_ROOT / 'models' / 'saved_models'

TARGET_SCORE_METHOD = 'mean'
TARGET_FPR = 0.20
CLASS_NAMES = ['Normal', 'Anomaly']
FIGURE_SUBDIR = 'multi_model_fpr20'
TABLE_SUBDIR = 'ecnn_threshold_experiments'

# Optional manual overrides (set to exact zip path if auto-discovery picks wrong files)
NORMAL_ZIP_OVERRIDE = Path("/content/drive/MyDrive/symAD-ECNN/data/val_fast.zip")
ANOMALY_ZIP_OVERRIDE = Path("/content/drive/MyDrive/symAD-ECNN/data/test_fast.zip")

# Extraction root on local disk for faster IO
LOCAL_EXTRACT_ROOT = Path('/content/local_eval_data') if IN_COLAB else (REPO_ROOT / '.local_eval_data')

# Model config: checkpoint search + model type
MODEL_CONFIGS = [
    {
        'key': 'ecnn_opt',
        'display_name': 'ECNN-AE (Optimized)',
        'model_type': 'ecnn',
        'checkpoint_dirs': ['ecnn_optimized', '.'],
        'checkpoint_patterns': ['ecnn_optimized_best.pth', 'ecnn_best.pth', '*ecnn*optimized*best*.pth', '*ecnn*best*.pth'],
    },
    {
        'key': 'cnn_large',
        'display_name': 'CNN-AE Large',
        'model_type': 'cnn_large',
        'checkpoint_dirs': ['cnn_ae_large'],
        'checkpoint_patterns': ['cnn_large_best.pth', '*cnn*large*best*.pth'],
    },
    {
        'key': 'cnn_aug',
        'display_name': 'CNN-AE Augmented',
        'model_type': 'cnn_base',
        'checkpoint_dirs': ['cnn_ae_augmented'],
        'checkpoint_patterns': ['cnn_aug_best.pth', '*cnn*aug*best*.pth'],
    },
    {
        'key': 'cnn_base',
        'display_name': 'CNN-AE',
        'model_type': 'cnn_base',
        'checkpoint_dirs': ['cnn_ae'],
        'checkpoint_patterns': ['cnn_best.pth', '*cnn*best*.pth'],
    },
    {
        'key': 'resnet_ae',
        'display_name': 'ResNet-AE',
        'model_type': 'resnet_frozen',
        'checkpoint_dirs': ['resnet_ae'],
        'checkpoint_patterns': ['resnet_best.pth', '*resnet*best*.pth'],
    },
    {
        'key': 'resnet_ft',
        'display_name': 'ResNet Finetuned (partial)',
        'model_type': 'resnet_finetuned_partial',
        'checkpoint_dirs': ['resnet_finetuned_partial', 'resnet_finetuned_full', 'resnet_finetuned_none'],
        'checkpoint_patterns': ['resnet_best.pth', '*resnet*best*.pth'],
    },
]

print(f"Project root: {PROJECT_ROOT}")
print(f"Data root: {DATA_ROOT}")
print(f"Models root: {MODELS_ROOT}")
print(f"Figure output: {EVALUATIONS_ROOT / 'figures' / FIGURE_SUBDIR}")
print(f"Table output: {EVALUATIONS_ROOT / 'tables' / TABLE_SUBDIR}")

Project root: /content/drive/MyDrive/symAD-ECNN
Data root: /content/drive/MyDrive/symAD-ECNN/data
Models root: /content/drive/MyDrive/symAD-ECNN/models/saved_models
Figure output: /content/drive/MyDrive/symAD-ECNN/evaluations/figures/multi_model_fpr20
Table output: /content/drive/MyDrive/symAD-ECNN/evaluations/tables/ecnn_threshold_experiments


In [ ]:
# Helper functions and model definitions
import torch.nn.functional as F
from PIL import Image

from model_defs import CNNAutoencoder, LargeCNNAutoencoder, ResNetAutoencoder
from eval_common import (
    find_files,
    extract_zip,
    resolve_checkpoint_path as resolve_checkpoint_path_common,
    get_state_dict as get_state_dict_common,
    compute_reconstruction_errors,
)

class SliceDataset(Dataset):
    def __init__(self, files: List[Path], mode: str = 'grayscale'):
        self.files = files
        self.mode = mode
        self.resnet_transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225],
            ),
        ])

    def __len__(self):
        return len(self.files)

    def _load_slice(self, path: Path) -> np.ndarray:
        if path.suffix.lower() == '.npy':
            arr = np.load(path)
        else:
            arr = np.array(Image.open(path).convert('L'))
            arr = arr.astype(np.float32) / 255.0
        arr = arr.astype(np.float32)
        if arr.max() > 1.0:
            arr = arr / 255.0
        if arr.ndim == 3:
            arr = arr.squeeze()
        return arr

    def __getitem__(self, idx):
        arr = self._load_slice(self.files[idx])
        gray = torch.from_numpy(arr).float().unsqueeze(0)
        if gray.shape[-2:] != (128, 128):
            gray = F.interpolate(gray.unsqueeze(0), size=(128, 128), mode='bilinear', align_corners=False).squeeze(0)

        if self.mode == 'resnet':
            img_uint8 = (gray.squeeze(0).numpy() * 255.0).clip(0, 255).astype(np.uint8)
            rgb = np.stack([img_uint8, img_uint8, img_uint8], axis=-1)
            inp = self.resnet_transform(rgb)
            tgt = gray
            return inp, tgt

        return gray, gray
def discover_zip(zip_override: Optional[str], target: str) -> Path:
    if zip_override is not None:
        z = Path(zip_override)
        if not z.exists():
            raise FileNotFoundError(f'Override zip not found: {z}')
        return z

    all_zips = sorted(DATA_ROOT.rglob('*.zip'))
    if not all_zips:
        raise FileNotFoundError(f'No zip files found under {DATA_ROOT}')

    def score_zip(p: Path) -> int:
        n = p.name.lower()
        s = 0
        if target == 'normal':
            if 'val' in n: s += 5
            if 'ixi' in n: s += 5
            if 'normal' in n: s += 4
            if 'train' in n: s += 1
            if 'brats' in n: s -= 10
        else:
            if 'brats' in n: s += 8
            if 'anomaly' in n or 'tumor' in n: s += 6
            if 'test' in n: s += 3
            if 'ixi' in n: s -= 6
            if 'val' in n: s -= 3
        if 'fast' in n: s += 1
        return s

    ranked = sorted(all_zips, key=lambda p: (score_zip(p), p.name.lower()), reverse=True)
    best = ranked[0]
    print(f"Selected {target} zip: {best}")
    return best
def resolve_checkpoint_path(cfg: Dict) -> Path:
    chosen = resolve_checkpoint_path_common(cfg, MODELS_ROOT)
    print(f"{cfg['display_name']} checkpoint: {chosen}")
    return chosen

def load_model_for_config(cfg: Dict, device: str):
    ckpt_path = resolve_checkpoint_path(cfg)

    if cfg['model_type'] == 'ecnn':
        model, info = get_model_for_inference(ckpt_path, device)
        return model, ckpt_path

    checkpoint = torch.load(ckpt_path, map_location=device)
    state_dict = get_state_dict_common(checkpoint)

    if cfg['model_type'] == 'cnn_large':
        latent_dim = int(state_dict['fc_encode.weight'].shape[0]) if 'fc_encode.weight' in state_dict else 512
        model = LargeCNNAutoencoder(latent_dim=latent_dim)
    elif cfg['model_type'] == 'cnn_base':
        latent_dim = int(state_dict['fc_encode.weight'].shape[0]) if 'fc_encode.weight' in state_dict else 256
        model = CNNAutoencoder(latent_dim=latent_dim)
    elif cfg['model_type'] == 'resnet_frozen':
        model = ResNetAutoencoder(finetune_strategy='none')
    elif cfg['model_type'] == 'resnet_finetuned_partial':
        model = ResNetAutoencoder(finetune_strategy='partial')
    else:
        raise ValueError(f"Unsupported model type: {cfg['model_type']}")

    model.load_state_dict(state_dict, strict=False)
    model = model.to(device)
    model.eval()
    return model, ckpt_path

def compute_errors(model: nn.Module, dataloader: DataLoader, device: str) -> np.ndarray:
    return compute_reconstruction_errors(model, dataloader, device)

def summarize_scores(scores: np.ndarray) -> Dict[str, float]:
    return {
        'count': int(scores.size),
        'mean': float(np.mean(scores)),
        'std': float(np.std(scores)),
        'min': float(np.min(scores)),
        'max': float(np.max(scores)),
    }

print('Helper functions ready.')

Helper functions ready.


In [25]:
# Prepare data from zip files and evaluate all models
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

normal_zip = discover_zip(NORMAL_ZIP_OVERRIDE, target='normal')
anomaly_zip = discover_zip(ANOMALY_ZIP_OVERRIDE, target='anomaly')

normal_dir = extract_zip(normal_zip, LOCAL_EXTRACT_ROOT / 'normal')
anomaly_dir = extract_zip(anomaly_zip, LOCAL_EXTRACT_ROOT / 'anomaly')

normal_files = find_files(normal_dir)
anomaly_files = find_files(anomaly_dir)

print(f'Normal files: {len(normal_files)} from {normal_dir}')
print(f'Anomaly files: {len(anomaly_files)} from {anomaly_dir}')
if len(normal_files) == 0 or len(anomaly_files) == 0:
    raise RuntimeError('No data files found after extraction. Verify zip contents.')

model_summaries = []
roc_payload = []
failed_models = []

for cfg in MODEL_CONFIGS:
    print(f"\n=== {cfg['display_name']} ===")
    try:
        model, checkpoint_path = load_model_for_config(cfg, device)
        mode = 'resnet' if 'resnet' in cfg['model_type'] else 'grayscale'

        normal_ds = SliceDataset(normal_files, mode=mode)
        anomaly_ds = SliceDataset(anomaly_files, mode=mode)

        normal_loader = DataLoader(normal_ds, batch_size=32, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())
        anomaly_loader = DataLoader(anomaly_ds, batch_size=32, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())

        normal_scores = compute_errors(model, normal_loader, device)
        anomaly_scores = compute_errors(model, anomaly_loader, device)

        threshold = threshold_from_normal_scores(normal_scores, target_fpr=TARGET_FPR)
        y_true = np.concatenate([np.zeros(len(normal_scores), dtype=int), np.ones(len(anomaly_scores), dtype=int)])
        y_scores = np.concatenate([normal_scores, anomaly_scores])
        metrics = compute_full_metrics(y_true, y_scores, threshold)
        metrics['threshold'] = float(threshold)

        y_pred = (y_scores >= threshold).astype(int)

        fig_counts, ax_counts = plot_confusion_matrix(
            y_true, y_pred, class_names=CLASS_NAMES,
            title=f"{cfg['display_name']} - Confusion Matrix (Counts)",
            normalize=False,
        )
        save_figure(fig_counts, f"{cfg['key']}_confusion_counts_fpr20", subdir=FIGURE_SUBDIR)

        fig_norm, ax_norm = plot_confusion_matrix(
            y_true, y_pred, class_names=CLASS_NAMES,
            title=f"{cfg['display_name']} - Confusion Matrix (Normalized)",
            normalize=True,
        )
        save_figure(fig_norm, f"{cfg['key']}_confusion_normalized_fpr20", subdir=FIGURE_SUBDIR)

        fig_roc, ax_roc = plot_roc_curve(
            y_true, y_scores, auroc=metrics['auroc'],
            title=f"{cfg['display_name']} - ROC Curve",
            label=f"{cfg['display_name']} (AUROC={metrics['auroc']:.4f})",
        )
        save_figure(fig_roc, f"{cfg['key']}_roc_fpr20", subdir=FIGURE_SUBDIR)

        model_summaries.append({
            'model_key': cfg['key'],
            'model_name': cfg['display_name'],
            'checkpoint_path': str(checkpoint_path),
            'score_method': TARGET_SCORE_METHOD,
            'threshold_method': 'fpr',
            'target_fpr': TARGET_FPR,
            'threshold': metrics['threshold'],
            'accuracy': metrics['accuracy'],
            'precision': metrics['precision'],
            'recall': metrics['recall'],
            'specificity': metrics['specificity'],
            'f1_score': metrics['f1_score'],
            'fpr': metrics['fpr'],
            'fnr': metrics['fnr'],
            'auroc': metrics['auroc'],
            'auprc': metrics['auprc'],
            'tp': metrics['tp'],
            'tn': metrics['tn'],
            'fp': metrics['fp'],
            'fn': metrics['fn'],
            'normal_count': int(len(normal_scores)),
            'anomaly_count': int(len(anomaly_scores)),
            'normal_mean': float(normal_scores.mean()),
            'normal_std': float(normal_scores.std()),
            'anomaly_mean': float(anomaly_scores.mean()),
            'anomaly_std': float(anomaly_scores.std()),
        })

        roc_payload.append({
            'y_true': y_true,
            'y_scores': y_scores,
            'label': cfg['display_name'],
            'auroc': metrics['auroc'],
        })

        # Persist raw error arrays for reuse
        model_result_dir = EVALUATIONS_ROOT / 'json' / 'multi_model_fpr20_errors' / cfg['key']
        model_result_dir.mkdir(parents=True, exist_ok=True)
        np.save(model_result_dir / 'normal_errors.npy', normal_scores)
        np.save(model_result_dir / 'anomaly_errors.npy', anomaly_scores)

    except Exception as exc:
        print(f"WARNING: Skipping {cfg['display_name']} -> {exc}")
        failed_models.append({'model': cfg['display_name'], 'reason': str(exc)})

if failed_models:
    print('\nSkipped models:')
    for item in failed_models:
        print(f" - {item['model']}: {item['reason']}")
else:
    print('\nAll requested models evaluated successfully.')

Using device: cuda
Normal files: 3652 from /content/local_eval_data/normal
Anomaly files: 7794 from /content/local_eval_data/anomaly

=== ECNN-AE (Optimized) ===
ECNN-AE (Optimized) checkpoint: /content/drive/MyDrive/symAD-ECNN/models/saved_models/ecnn_optimized/ecnn_optimized_best.pth
Detected ECNNAutoencoderV3 checkpoint. Using latent_dim=1024.
Model weights loaded from: /content/drive/MyDrive/symAD-ECNN/models/saved_models/ecnn_optimized/ecnn_optimized_best.pth


Computing reconstruction errors:   0%|          | 0/115 [00:00<?, ?it/s]

Computing reconstruction errors:   0%|          | 0/244 [00:00<?, ?it/s]

Figure saved: /content/drive/MyDrive/symAD-ECNN/evaluations/figures/multi_model_fpr20/ecnn_opt_confusion_counts_fpr20.png
Figure saved: /content/drive/MyDrive/symAD-ECNN/evaluations/figures/multi_model_fpr20/ecnn_opt_confusion_normalized_fpr20.png
Figure saved: /content/drive/MyDrive/symAD-ECNN/evaluations/figures/multi_model_fpr20/ecnn_opt_roc_fpr20.png

=== CNN-AE Large ===
CNN-AE Large checkpoint: /content/drive/MyDrive/symAD-ECNN/models/saved_models/cnn_ae_large/cnn_large_best.pth


Computing reconstruction errors:   0%|          | 0/115 [00:00<?, ?it/s]

...

Computing reconstruction errors:   0%|          | 0/244 [00:00<?, ?it/s]

Figure saved: /content/drive/MyDrive/symAD-ECNN/evaluations/figures/multi_model_fpr20/resnet_ae_confusion_counts_fpr20.png
Figure saved: /content/drive/MyDrive/symAD-ECNN/evaluations/figures/multi_model_fpr20/resnet_ae_confusion_normalized_fpr20.png
Figure saved: /content/drive/MyDrive/symAD-ECNN/evaluations/figures/multi_model_fpr20/resnet_ae_roc_fpr20.png

=== ResNet Finetuned (partial) ===
ResNet Finetuned (partial) checkpoint: /content/drive/MyDrive/symAD-ECNN/models/saved_models/resnet_finetuned_partial/resnet_best.pth


Computing reconstruction errors:   0%|          | 0/115 [00:00<?, ?it/s]

Computing reconstruction errors:   0%|          | 0/244 [00:00<?, ?it/s]

Figure saved: /content/drive/MyDrive/symAD-ECNN/evaluations/figures/multi_model_fpr20/resnet_ft_confusion_counts_fpr20.png
Figure saved: /content/drive/MyDrive/symAD-ECNN/evaluations/figures/multi_model_fpr20/resnet_ft_confusion_normalized_fpr20.png
Figure saved: /content/drive/MyDrive/symAD-ECNN/evaluations/figures/multi_model_fpr20/resnet_ft_roc_fpr20.png

All requested models evaluated successfully.


In [26]:
# Metrics table display and export
from IPython.display import display

if model_summaries:
    metrics_df = pd.DataFrame(model_summaries)
    metrics_df = metrics_df.sort_values('auroc', ascending=False).reset_index(drop=True)

    display(
        metrics_df.style.format({
            'target_fpr': '{:.2f}',
            'threshold': '{:.6f}',
            'accuracy': '{:.4f}',
            'precision': '{:.4f}',
            'recall': '{:.4f}',
            'specificity': '{:.4f}',
            'f1_score': '{:.4f}',
            'fpr': '{:.4f}',
            'fnr': '{:.4f}',
            'auroc': '{:.4f}',
            'auprc': '{:.4f}',
            'normal_mean': '{:.6f}',
            'normal_std': '{:.6f}',
            'anomaly_mean': '{:.6f}',
            'anomaly_std': '{:.6f}',
        })
    )

    csv_path = save_csv(metrics_df, 'multi_model_fpr20_metrics.csv', subdir=TABLE_SUBDIR)
    md_path = save_markdown_table(
        metrics_df,
        'multi_model_fpr20_metrics.md',
        title='Multi-Model Metrics (Score=mean, Threshold=fpr, Param=0.2)',
        subdir=TABLE_SUBDIR,
    )

    print(f'CSV saved: {csv_path}')
    print(f'Markdown saved: {md_path}')
else:
    print('No successful evaluations. Verify score arrays exist for all models.')

,model_key,model_name,checkpoint_path,score_method,threshold_method,target_fpr,threshold,accuracy,precision,recall,specificity,f1_score,fpr,fnr,auroc,auprc,tp,tn,fp,fn,normal_count,anomaly_count,normal_mean,normal_std,anomaly_mean,anomaly_std
0,resnet_ae,ResNet-AE,/content/drive/MyDrive/symAD-ECNN/models/saved_models/resnet_ae/resnet_best.pth,mean,fpr,0.20,0.012339,0.7776,0.8911,0.7671,0.7998,0.8245,0.2002,0.2329,0.8748,0.9326,5979,2921,731,1815,3652,7794,0.009366,0.003869,0.017952,0.007421
1,ecnn_opt,ECNN-AE (Optimized),/content/drive/MyDrive/symAD-ECNN/models/saved_models/ecnn_optimized/ecnn_optimized_best.pth,mean,fpr,0.20,0.003935,0.6586,0.8633,0.5924,0.7998,0.7026,0.2002,0.4076,0.8109,0.8813,4617,2921,731,3177,3652,7794,0.002859,0.001547,0.005013,0.002547
2,cnn_large,CNN-AE Large,/content/drive/MyDrive/symAD-ECNN/models/saved_models/cnn_ae_large/cnn_large_best.pth,mean,fpr,0.20,0.006904,0.6124,0.8483,0.5246,0.7998,0.6483,0.2002,0.4754,0.7803,0.8589,4089,2921,731,3705,3652,7794,0.005143,0.002446,0.007888,0.003277
3,cnn_base,CNN-AE,/content/drive/MyDrive/symAD-ECNN/models/saved_models/cnn_ae/cnn_best.pth,mean,fpr,0.20,0.007245,0.5811,0.8361,0.4786,0.7998,0.6087,0.2002,0.5214,0.7617,0.8479,3730,2921,731,4064,3652,7794,0.005470,0.002508,0.008051,0.003433
4,resnet_ft,ResNet Finetuned (partial),/content/drive/MyDrive/symAD-ECNN/models/saved_models/resnet_finetuned_partial/resnet_best.pth,mean,fpr,0.20,0.008822,0.5619,0.8277,0.4505,0.7998,0.5834,0.2002,0.5495,0.7398,0.8355,3511,2921,731,4283,3652,7794,0.006519,0.002985,0.009301,0.003968
5,cnn_aug,CNN-AE Augmented,/content/drive/MyDrive/symAD-ECNN/models/saved_models/cnn_ae_augmented/cnn_aug_best.pth,mean,fpr,0.20,0.008570,0.5171,0.8040,0.3847,0.7998,0.5204,0.2002,0.6153,0.7072,0.8023,2998,2921,731,4796,3652,7794,0.006328,0.003007,0.008342,0.003319


CSV saved: /content/drive/MyDrive/symAD-ECNN/evaluations/tables/ecnn_threshold_experiments/multi_model_fpr20_metrics.csv
Markdown table saved: /content/drive/MyDrive/symAD-ECNN/evaluations/tables/ecnn_threshold_experiments/multi_model_fpr20_metrics.md
CSV saved: /content/drive/MyDrive/symAD-ECNN/evaluations/tables/ecnn_threshold_experiments/multi_model_fpr20_metrics.csv
Markdown saved: /content/drive/MyDrive/symAD-ECNN/evaluations/tables/ecnn_threshold_experiments/multi_model_fpr20_metrics.md


In [27]:
# Combined ROC plot
if roc_payload:
    fig, ax = plot_multiple_roc_curves(
        roc_payload,
        title='ROC Comparison (All Models, Score=mean, Threshold=fpr@0.2)',
    )
    save_figure(fig, 'multi_model_fpr20_roc_comparison', subdir=FIGURE_SUBDIR)
    print('Saved combined ROC figure.')
else:
    print('No ROC payload available.')

Figure saved: /content/drive/MyDrive/symAD-ECNN/evaluations/figures/multi_model_fpr20/multi_model_fpr20_roc_comparison.png
Saved combined ROC figure.


## Notes

- This notebook works directly from ZIP data + checkpoint `.pth` files on Drive.
- If auto-selected ZIP files are incorrect, set `NORMAL_ZIP_OVERRIDE` and/or `ANOMALY_ZIP_OVERRIDE` in the configuration cell.
- Extracted files are cached under `LOCAL_EXTRACT_ROOT` for faster evaluation.
- Raw per-model error arrays are saved to `evaluations/json/multi_model_fpr20_errors/<model_key>/`.
- Thresholding policy is fixed to: **Score = mean**, **Threshold = fpr**, **Param = 0.2**.